In [6]:
%load_ext autoreload
%autoreload 2


import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
import matplotlib.pyplot as plt
from scripts.main import load_graph_and_communities
from collections import Counter
import pandas as pd

datasets = ['Algebra', 'Geometry', 'Music-Rev', 'Restaurants-Rev', 'Bars-Rev', 'contact-primary-school', 'contact-high-school']

summary = []
dup_rows = []

for d in datasets:
    hg, communities = load_graph_and_communities(d, tau=2, verbose=True, source=False)
    unique_node_sets = set(frozenset(v) for v in hg.hyperedges.values())
    print(f"Dataset: {d}, len(hg): {len(hg.hyperedges)}, len(unique_node_sets): {len(unique_node_sets)}, duplicates: {len(hg.hyperedges) - len(unique_node_sets)}")

    edge_sets = [frozenset(e) for e in hg.hyperedges.values()]
    total = len(edge_sets)
    cnt = Counter(edge_sets)
    unique = len(cnt)
    duplicates = total - unique
    summary.append({
        "dataset": d,
        "total_hyperedges": total,
        "unique_hyperedge_sets": unique,
        "duplicate_hyperedges": duplicates,
        "duplicate_fraction": duplicates / total if total else 0,
    })
    for s, c in cnt.items():
        if c > 1:
            dup_rows.append({"dataset": d, "hyperedge": ",".join(sorted(s)), "count": c})

summary_df = pd.DataFrame(summary).sort_values("duplicate_hyperedges", ascending=False).reset_index(drop=True)
duplicates_df = pd.DataFrame(dup_rows).sort_values(["dataset", "count"], ascending=[True, False]).reset_index(drop=True)

display(summary_df)
if not duplicates_df.empty:
    display(duplicates_df)
else:
    print("No duplicate hyperedges found across datasets.")


Loaded dataset 'Algebra' (edge-labeled) with 1268 hyperedges and 10 communities
  communities sizes: min=2, max=88, avg=42.300
Dataset: Algebra, len(hg): 1268, len(unique_node_sets): 980, duplicates: 288
Loaded dataset 'Geometry' (edge-labeled) with 1193 hyperedges and 21 communities
  communities sizes: min=1, max=137, avg=27.619
Dataset: Geometry, len(hg): 1193, len(unique_node_sets): 888, duplicates: 305
Loaded dataset 'Music-Rev' (edge-labeled) with 694 hyperedges and 24 communities
  communities sizes: min=2, max=361, avg=46.083
Dataset: Music-Rev, len(hg): 694, len(unique_node_sets): 686, duplicates: 8
Loaded dataset 'Restaurants-Rev' (edge-labeled) with 601 hyperedges and 7 communities
  communities sizes: min=11, max=177, avg=80.714
Dataset: Restaurants-Rev, len(hg): 601, len(unique_node_sets): 594, duplicates: 7
Loaded dataset 'Bars-Rev' (edge-labeled) with 1194 hyperedges and 37 communities
  communities sizes: min=4, max=314, avg=33.351
Dataset: Bars-Rev, len(hg): 1194, len(

,dataset,total_hyperedges,unique_hyperedge_sets,duplicate_hyperedges,duplicate_fraction
0,Geometry,1193,888,305,0.255658
1,Algebra,1268,980,288,0.227129
2,Music-Rev,694,686,8,0.011527
3,Restaurants-Rev,601,594,7,0.011647
4,Bars-Rev,1194,1188,6,0.005025
5,contact-primary-school,12704,12704,0,0.000000
6,contact-high-school,7818,7818,0,0.000000


,dataset,hyperedge,count
0,Algebra,"135,197",10
1,Algebra,"102,33,71",10
2,Algebra,"52,58,68",8
3,Algebra,"90,95",8
4,Algebra,"77,91,93",8
...,...,...,...
270,Restaurants-Rev,"35,44",2
271,Restaurants-Rev,"24,71,72",2
272,Restaurants-Rev,"113,81",2
273,Restaurants-Rev,"182,359",2


In [ ]:
import statistics

import matplotlib.pyplot as plt

from scripts.main import load_graph_and_communities, run_configured_sicp_intermediates


hg,  communities = load_graph_and_communities("Geometry", tau=2, verbose=True, source=False)


BETA = 0.2
T = 25
RUNS = 10
RNG_SEED = 175
SEED_ITERATIONS = 200



edge_nodes = set().union(*hg.hyperedges.values()) if hg.hyperedges else set()
extra_nodes = hg.nodes - edge_nodes

res_old = run_configured_sicp_intermediates(
    hg,
    seed_iterations=SEED_ITERATIONS,
    beta=BETA,
    T=T,
    runs=RUNS,
    rng_seed=RNG_SEED,
)


N = len(hg.nodes)                      # total number of nodes in the simulation
timesteps = list(range(T + 1))
prevalence = [x / N for x in res_old]  # convert mean infected count -> mean prevalence

plt.figure(figsize=(8, 5))
plt.plot(timesteps, prevalence, marker="o")
plt.xlabel("Timestep")
plt.ylabel("Prevalence")
plt.title("SICP prevalence over time")
plt.ylim(0, 1)
plt.xlim(0, T)
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

csv_path = Path("../nkernokkel") / "Algebra_comprehensive_results.csv"
df = pd.read_csv(csv_path)

# Pick a time column if present; otherwise use row index
time_candidates = ["time", "timestep", "step", "t", "iteration"]
time_col = next((c for c in time_candidates if c in df.columns.str.lower()), None)

if time_col is None:
    # map lowercase match back to actual column names
    lower_to_actual = {c.lower(): c for c in df.columns}
    matched = next((c for c in time_candidates if c in lower_to_actual), None)
    time_col = lower_to_actual[matched] if matched else None

x = df[time_col] if time_col else df.index

# Plot all numeric columns except time
numeric_cols = df.select_dtypes(include="number").columns.tolist()
if time_col in numeric_cols:
    numeric_cols.remove(time_col)

# uses existing variables: df, col, time_col
agg = df.groupby(["strategy", "p", time_col])[col].mean().reset_index()

plt.figure(figsize=(10, 6))
for (strategy, p), g in agg.groupby(["strategy", "p"]):
    g = g.sort_values(time_col)
    plt.plot(g[time_col], g[col], marker="o", label=f"{strategy} | p={p}")

plt.xlabel(time_col)
plt.ylabel(col)
plt.title("Mean infected over time by strategy and removal fraction")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()
